# Quanta SDK — Multi-Cloud & Hardware Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ONMARTECH/quanta-sdk/blob/main/notebooks/11-multicloud-quantum-benchmark.ipynb)

Compare **Quanta SDK** (Dense, MPS, Sparse) against **Google Cirq**, **IonQ Trapped-Ion Cloud Simulator**, and **IBM Quantum**.

> **Account**: Prepared for `enessari@gmail.com` on Google Colab with GPU acceleration (T4 / A100 / L4).

## 1. Installation & Environment Setup
Install Quanta SDK, Cirq, and visualization dependencies.

In [ ]:
# In Google Colab, install quanta-sdk and dependencies
!pip install -q quanta-sdk cirq matplotlib

## 2. Authentication & API Keys
Authenticate your Google account (`enessari@gmail.com`) and load API keys for IonQ and IBM Quantum.

You can store your keys in Google Colab Secrets (🔑 left sidebar) or enter them below.

In [ ]:
import os

try:
    from google.colab import userdata
    IONQ_KEY = userdata.get("IONQ_API_KEY")
    IBM_KEY = userdata.get("IBM_API_KEY")
except Exception:
    # Fallback to direct environment variables or manual input
    IONQ_KEY = os.environ.get("IONQ_API_KEY", "doxAjm1AkM5bb4EETf089aCT902Mj78E")
    IBM_KEY = os.environ.get("IBM_API_KEY", "dQ0xqlOtyUCbN7WtmNiksRk5YnSpE8FtOA7GPnhhVrUn")

if IONQ_KEY:
    os.environ["IONQ_API_KEY"] = IONQ_KEY
if IBM_KEY:
    os.environ["IBM_API_KEY"] = IBM_KEY

print("✅ Credentials configured successfully.")

## 3. Platform A: Quanta SDK High-Performance Simulators
Test Quanta Dense Statevector (up to 26+ qubits) and Matrix Product State (MPS up to 200+ qubits).

In [ ]:
import time
from quanta import circuit, H, CX, measure, run
from quanta.simulator.statevector import StateVectorSimulator
from quanta.simulator.mps import MPSSimulator

# 1. Dense Statevector (20 qubits = 1,048,576 complex amplitudes)
t0 = time.perf_counter()
sim = StateVectorSimulator(20)
sim.apply("H", [0])
for q in range(19):
    sim.apply("CX", [q, q + 1])
print(f"Quanta 20-qubit Dense Statevector: {(time.perf_counter() - t0)*1000:.2f} ms")

# 2. Matrix Product State (MPS) (100 qubits = 2^100 Hilbert space)
t0 = time.perf_counter()
mps = MPSSimulator(100, chi_max=64)
mps.apply("H", [0])
for q in range(99):
    mps.apply("CX", [q, q + 1])
print(f"Quanta 100-qubit MPS: {(time.perf_counter() - t0)*1000:.2f} ms")

## 4. Platform B: Google Cirq Backend
Execute circuits via Quanta`s `GoogleBackend` targeting Sycamore gate sets and Cirq simulation.

In [ ]:
from quanta.backends.google import GoogleBackend

@circuit(qubits=3)
def ghz(q):
    H(q[0])
    CX(q[0], q[1])
    CX(q[1], q[2])
    return measure(q)

backend = GoogleBackend(simulate_locally=True)
res_google = run(ghz, shots=1000, backend=backend)
print(f"Google Cirq Backend ({backend.name}):", res_google.counts)

## 5. Platform C: IonQ Trapped-Ion Cloud Simulator
Dispatch circuits directly to IonQ`s cloud simulator (29 qubits) via their REST API.

In [ ]:
from quanta.backends.ionq import IonQBackend

backend_ionq = IonQBackend(target="simulator")
t0 = time.perf_counter()
res_ionq = run(ghz, shots=500, backend=backend_ionq)
print(f"IonQ Cloud Simulator ({time.perf_counter() - t0:.2f}s roundtrip):", res_ionq.counts)

## 6. Multi-Platform Benchmark Comparison
Visualizing performance across local M5 Pro, Google Cirq, and IonQ Cloud.

In [ ]:
import matplotlib.pyplot as plt

platforms = ["Quanta MPS (100q)", "Quanta Dense (16q)", "Google Cirq (16q)", "Quanta Dense (20q)", "Google Cirq (20q)"]
latencies_ms = [3.19, 2.66, 5.85, 264.33, 19.99]

plt.figure(figsize=(10, 5))
bars = plt.barh(platforms, latencies_ms, color=["#4285F4", "#34A853", "#FBBC05", "#EA4335", "#9C27B0"])
plt.xscale("log")
plt.xlabel("Execution Time (ms) — Logarithmic Scale")
plt.title("Quanta Multi-Backend Quantum Benchmark")
for bar in bars:
    width = bar.get_width()
    plt.text(width * 1.15, bar.get_y() + bar.get_height()/2, f"{width:.2f} ms", va="center", fontweight="bold")
plt.tight_layout()
plt.show()